In [0]:
# ============================================================
# CONFIGURE ME: set your catalog and schema before running
# ============================================================
catalog = "my_catalog"
schema  = "lr_genie_demo"

# ============================================================
# Setup - no need to edit below this line
# ============================================================

# Set context
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

# Create temp view and DA map of meta keys/values
spark.sql("""
    CREATE OR REPLACE TEMP VIEW user_info AS
    SELECT map_from_arrays(collect_list(key), collect_list(value))
    FROM meta
""")

spark.sql("DECLARE OR REPLACE DA MAP<STRING,STRING>")
spark.sql("SET VAR DA = (SELECT * FROM user_info)")

# Set default schema for the user
spark.sql("USE SCHEMA IDENTIFIER(DA.schema_name)")

# Copy tables from our seed tables
for table in ["opportunities", "orders", "customers"]:
    spark.sql(f"""
        CREATE OR REPLACE TABLE ca_{table} AS
        SELECT * FROM {catalog}.{schema}.{table}_ca
    """)
    print(f"✅ Created ca_{table}")

# Create products table (hardcoded data, no seed CSV needed)
spark.sql("DROP TABLE IF EXISTS ca_products")
spark.sql("""
    CREATE TABLE ca_products (
        productid   STRING,
        productname STRING,
        listprice   FLOAT
    )
""")
spark.sql("""
    INSERT INTO ca_products (productid, productname, listprice) VALUES
        ('JQ9322',  'TableSaw',   24.99),
        ('A802a',   'Mill',       10.17),
        ('EE05x',   'Press',      363.02),
        ('R9S10',   'Lathe',      43.83),
        ('BC111d',  'Drill',      63.28),
        ('W8931',   'Planer',     45.22),
        ('Q28R3',   'Grinder',    176.33),
        ('MDb304',  'CAD System', 502.80),
        ('OR307p',  'Kazoo',      0.99)
""")
print("✅ Created ca_products")

# Show user their catalog + schema
display(spark.sql("SELECT current_catalog() AS CourseCatalog, current_schema() AS YourSchema"))